# 📜 Notebook 1: One Big Log File (the BAD way)

**The problem:** Many systems (databases, message brokers, write-ahead logs) need to *append* records forever. The simplest design is one giant file — but that has nasty failure modes.

We'll build it, then watch it break.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/segmented-log
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 Approach: append everything to one file

In [1]:
import os, tempfile, time

WORKDIR = tempfile.mkdtemp(prefix='single_log_')
LOG = os.path.join(WORKDIR, 'data.log')

def append(record: bytes):
    # 'ab' = append, binary. Each record is length-prefixed (4 bytes) + payload.
    with open(LOG, 'ab') as f:
        f.write(len(record).to_bytes(4, 'big'))
        f.write(record)

def read_all():
    out = []
    with open(LOG, 'rb') as f:
        while True:
            hdr = f.read(4)
            if not hdr: break
            n = int.from_bytes(hdr, 'big')
            out.append(f.read(n))
    return out

for i in range(1000):
    append(f'event-{i}'.encode())

print('records:', len(read_all()))
print('file size (bytes):', os.path.getsize(LOG))


records: 1000
file size (bytes): 12890


## 🤔 What goes wrong as the log grows?

1. **Deleting old records is hard.** You can't `truncate from the front` of a file cheaply — you have to copy the whole thing.
2. **One huge file** is annoying to back up, ship between machines, and recover from corruption.
3. **Parallel reads/writes** all hit the same file.
4. **Crash recovery** must scan the entire file to find the last valid record.

Let's *show* problem #1: pretend we want to drop the oldest 500 records.

In [2]:
import shutil
t0 = time.perf_counter()
records = read_all()
kept = records[500:]
with open(LOG, 'wb') as f:
    for r in kept:
        f.write(len(r).to_bytes(4,'big')); f.write(r)
print(f'rewrote whole file in {time.perf_counter()-t0:.4f}s for only 500 deletions')
print('remaining:', len(read_all()))


rewrote whole file in 0.0015s for only 500 deletions
remaining: 500


Rewriting the whole file just to drop the head is **O(n)** in total bytes, every time. With a 50 GB log this is unusable.

👉 Next notebook: split the log into **segments** so we can delete old data by simply unlinking files.